In [ ]:
from pathlib import Path
import random
from typing import Iterator

from torch import Tensor
from torch.utils.data import IterableDataset
from torchvision.io import decode_image


class EpisodeDataset(IterableDataset):
    def __init__(self, root: str, n_way: int, k_shot: int, q_query: int,transform:torch.nn.Module|None=None) -> None:
        super().__init__()
        self.root = root
        self.n_way = n_way
        self.k_shot = k_shot
        self.q_query = q_query
        self.transform=transform

        # self.data:list[tuple[Path,int]]=[]
        # [(WindowsPath('omniglot/python/images_background/Alphabet_of_the_Magi/character01/0709_01.png'), 0)]

        self.character_to_images: dict[str, list[str]] = {}

        self._load()

    def _load(self) -> None:
        root_path = Path(self.root)

        for language_path in root_path.iterdir():
            if not language_path.is_dir():
                continue

            for character_path in language_path.iterdir():
                if not character_path.is_dir():
                    continue

                for image_path in character_path.iterdir():
                    if not image_path.is_file():
                        continue

                    # self.data.append((image_path,class_i))

                    self.character_to_images.setdefault(character_path.as_posix(), []).append(image_path.as_posix())

        # print(self.character_to_images[r'omniglot/python/images_background/Alphabet_of_the_Magi/character01'])

    def _load_image(self,path_str:str)->Tensor:
        image=decode_image(path_str)

        if self.transform:
            image=self.transform(image)

        image:Tensor=image.float()/255

        return image
    
    def __iter__(self)  -> Iterator[tuple[list[Tensor], list[int], list[Tensor], list[int]]]:
        selected_character_strs = random.sample(tuple(self.character_to_images.keys()), self.n_way)

        support:list[Tensor] = []
        support_labels:list[int] = []
        query:list[Tensor] = []
        query_labels:list[int] = []

        for i, character in enumerate(selected_character_strs):
            image_strs: list[str] = random.sample(self.character_to_images[character], self.k_shot + self.q_query)

            images=[self._load_image(image_str) for image_str in image_strs]
            
            support += images[: self.k_shot]
            support_labels += [i] * self.k_shot
            query += images[self.k_shot :]
            query_labels += [i] * self.q_query

        yield support,support_labels,query,query_labels


EpisodeDataset(root='./omniglot/python/images_background', n_way=1, k_shot=1, q_query=1)

['omniglot/python/images_background/Alphabet_of_the_Magi/character01/0709_01.png', 'omniglot/python/images_background/Alphabet_of_the_Magi/character01/0709_02.png', 'omniglot/python/images_background/Alphabet_of_the_Magi/character01/0709_03.png', 'omniglot/python/images_background/Alphabet_of_the_Magi/character01/0709_04.png', 'omniglot/python/images_background/Alphabet_of_the_Magi/character01/0709_05.png', 'omniglot/python/images_background/Alphabet_of_the_Magi/character01/0709_06.png', 'omniglot/python/images_background/Alphabet_of_the_Magi/character01/0709_07.png', 'omniglot/python/images_background/Alphabet_of_the_Magi/character01/0709_08.png', 'omniglot/python/images_background/Alphabet_of_the_Magi/character01/0709_09.png', 'omniglot/python/images_background/Alphabet_of_the_Magi/character01/0709_10.png', 'omniglot/python/images_background/Alphabet_of_the_Magi/character01/0709_11.png', 'omniglot/python/images_background/Alphabet_of_the_Magi/character01/0709_12.png', 'omniglot/pytho